In [ ]:
import os, gc, math, time, wandb, random, logging
os.environ["NUMBA_NUM_THREADS"] = str(os.cpu_count())
os.environ["NUMBA_THREADING_LAYER"] = "tbb"

from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import polars as pl

from dotenv import load_dotenv

from tokenizers import Tokenizer

from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer

from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Sequence, NFKC, Lowercase, Strip

from tokenizers.processors import TemplateProcessing

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import optuna
from optuna.samplers import TPESampler

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold

from sentence_transformers import SentenceTransformer

from pacmap import PaCMAP
from hdbscan import HDBSCAN

from transformers import logging as hf_logging
from transformers.utils.logging import disable_progress_bar

In [ ]:
# Load datasets and encode label as `label_id`
TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

OPTION_COLS = ["A", "B", "C", "D", "E"]

LABEL_COL = "answer"
LABEL2ID = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

train_data = pl.read_csv(TRAIN_PATH)
test_data = pl.read_csv(TEST_PATH)
train_data = train_data.with_columns(
    pl.col(LABEL_COL).replace(LABEL2ID).cast(pl.Int8).alias("label_id")
)

train_data = train_data.with_columns(
    (
        pl.lit("Prompt : ") + pl.col("prompt")
        + pl.lit(" Options: ")
        + pl.concat_str(
            [pl.lit(f"{opt}) ") + pl.col(opt).fill_null(" ") for opt in OPTION_COLS],
            separator=" "
        )
    ).alias("mcq_query")
)

In [ ]:
# Configure training parameters, data, models, device, and seeds for reproducibility
EPOCHS = 10
N_SPLITS = 5

BATCH_SIZE = 8
ACCUMULATION_STEPS = 4

ID_COL = "id"
QUESTION_COL = "prompt"
OPTION_COLS  = ["A", "B", "C", "D", "E"]

# EMBED_MODEL = "baai/bge-m3"
EMBED_MODEL = "Qwen/Qwen3-Embedding-0.6B"

RANK_MODEL = "custom-architecture/BiLSTM-Cross-Attention-Classifier"
# RANK_MODEL = "custom-architecture/BiGRU-Cross-Attention-Classifier"

VOCAB_SIZE = 2000

NUM_ENCODER_LAYERS = 1
EMBEDDING_DIM = 64
HIDDEN_DIM = 64
DROPOUT = 0.4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SEED = 42

def set_seed(seed: int, is_final_run: bool = False) -> None:
    
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False if not is_final_run else True
    torch.backends.cudnn.deterministic = True if not is_final_run else False

set_seed(SEED, True)

# Create directory to store saved models
BEST_DIR = "best-cv-models"
os.makedirs(BEST_DIR, exist_ok=True)

# Configure logging levels to hide model-loading report
hf_logging.set_verbosity_error()

logging.getLogger("transformers").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)

disable_progress_bar()

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Configure HuggingFace API key
load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_READ_TOKEN") if os.getenv("HF_READ_TOKEN") else "" # type: ignore

# Configure WandB info
ist_now = datetime.now(ZoneInfo("Asia/Kolkata"))
WANDB_RUN_NAME = f"{RANK_MODEL.split('/')[-1]}_{ist_now:%Y-%m-%d_%H-%M-%S}_IST"


wandb.login(key=os.getenv("WANDB_API_KEY")) if os.getenv("HF_READ_TOKEN") else "" # type: ignore

In [ ]:
# Define DataLoader seeding function, and Generator for reproducibility
def seed_worker(worker_id: int = 42):
    
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_generator = torch.Generator()
train_generator.manual_seed(SEED)
val_generator = torch.Generator()
val_generator.manual_seed(SEED)

test_generator = torch.Generator()
test_generator.manual_seed(SEED)

In [ ]:
# Define custom Byte-Pair Tokenizer
class BPETokenizer:
    SPECIAL_TOKENS = [
        "[pad]",
        "[unk]",
        "[cls]",
        "[sep]",
        "[opt_a]",
        "[opt_b]",
        "[opt_c]",
        "[opt_d]",
        "[opt_e]"
    ]

    # Initilize BPETokenizer class
    def __init__(self, vocab_size: int = VOCAB_SIZE, min_frequency: int = 1):
        self.vocab_size = vocab_size
        self.min_frequency = min_frequency
        self.tokenizer = None

    # Build context to train the tokenizer class with
    @staticmethod
    def build_context(df: pl.DataFrame) -> list[str]:
        """Serialize DataFrame rows into plain training strings."""
        expr = pl.concat_str(
            [
                pl.col("prompt").fill_null("").cast(pl.Utf8),
                pl.lit(" [sep] "),
                pl.lit("[opt_a] "), pl.col("A").fill_null("").cast(pl.Utf8),
                pl.lit(" [opt_b] "), pl.col("B").fill_null("").cast(pl.Utf8),
                pl.lit(" [opt_c] "), pl.col("C").fill_null("").cast(pl.Utf8),
                pl.lit(" [opt_d] "), pl.col("D").fill_null("").cast(pl.Utf8),
                pl.lit(" [opt_e] "), pl.col("E").fill_null("").cast(pl.Utf8),
            ],
            separator=""
        ).alias("train_text")

        return df.select(expr)["train_text"].to_list()

    # Train tokenizer with training data
    def train(self, texts: list[str], save_path: str = "bpetokenizer.json") -> None:
        """Train the BPE tokenizer on a list of plain text strings."""
        tokenizer = Tokenizer(BPE(unk_token="[unk]"))

        tokenizer.normalizer = Sequence([
            NFKC(),
            Lowercase(),
            Strip()
        ])

        tokenizer.pre_tokenizer = Whitespace()

        trainer = BpeTrainer(
            vocab_size=self.vocab_size,
            min_frequency=self.min_frequency,
            special_tokens=self.SPECIAL_TOKENS,
            show_progress=False,
        )

        tokenizer.train_from_iterator(texts, trainer=trainer)

        cls_id = tokenizer.token_to_id("[cls]")
        sep_id = tokenizer.token_to_id("[sep]")
        pad_id = tokenizer.token_to_id("[pad]")
        
        tokenizer.enable_padding(pad_id=pad_id, pad_token="[pad]")

        tokenizer.post_processor = TemplateProcessing(
            single="[cls] $A [sep]",
            special_tokens=[
                ("[cls]", cls_id),
                ("[sep]", sep_id)
            ]
        )

        tokenizer.save(save_path)
        self.tokenizer = tokenizer

    def save(self, path: str) -> None:
        self._check_trained()
        self.tokenizer.save(path) # type: ignore

    @classmethod
    def load(cls, path: str) -> "BPETokenizer":
        """Load a previously saved tokenizer from disk."""
        instance = cls()
        instance.tokenizer = Tokenizer.from_file(path)
        return instance

    # Encode data with a traiened tokenizer
    def encode(self, data: dict | pl.DataFrame) -> dict | list[dict]:
        """Encode a single row dict or a full Polars DataFrame.

        Returns:
            - dict with keys (tokens, input_ids, attention_mask) for a single row
            - list[dict] for a DataFrame
        """
        self._check_trained()

        if isinstance(data, pl.DataFrame):
            return [self._encode_text(self._encode_row(row)) for row in data.to_dicts()]

        return self._encode_text(self._encode_row(data))

    def get_vocab_size(self) -> int:
        self._check_trained()
        return self.tokenizer.get_vocab_size() # type: ignore

    def token_to_id(self, token: str) -> int:
        self._check_trained()
        return self.tokenizer.token_to_id(token) # type: ignore

    # Internal helpers to encode data
    @staticmethod
    def _encode_row(row: dict) -> str:
        """Serialize a single row dict to the same format used during training."""
        return (
            f"{row['prompt']} [sep] "
            f"[opt_a] {row['A']} "
            f"[opt_b] {row['B']} "
            f"[opt_c] {row['C']} "
            f"[opt_d] {row['D']} "
            f"[opt_e] {row['E']}"
        )

    def _encode_text(self, text: str) -> dict:
        """Encode a single text string into token IDs."""
        enc = self.tokenizer.encode(text) # type: ignore
        return {
            "tokens": enc.tokens,
            "input_ids": enc.ids,
            "attention_mask": enc.attention_mask
        }

    def _check_trained(self) -> None:
        if self.tokenizer is None:
            raise RuntimeError("Tokenizer has not been trained yet. Call .train() first.")

In [ ]:
# Define custom MCQDataset class
class MCQDataset(Dataset):
    def __init__(self, df: pl.DataFrame, tokenizer: BPETokenizer, is_test: bool = False) -> None:
        self.tokenizer = tokenizer
        self.is_test = is_test
        
        # Encode columns independently
        self.prompts = [tokenizer._encode_text(text) for text in df["prompt"].to_list()]
        self.opt_a = [tokenizer._encode_text(text) for text in df["A"].to_list()]
        self.opt_b = [tokenizer._encode_text(text) for text in df["B"].to_list()]
        self.opt_c = [tokenizer._encode_text(text) for text in df["C"].to_list()]
        self.opt_d = [tokenizer._encode_text(text) for text in df["D"].to_list()]
        self.opt_e = [tokenizer._encode_text(text) for text in df["E"].to_list()]
        
        if not self.is_test:
            self.labels = df["label_id"].to_list()

    def __len__(self) -> int:
        return len(self.prompts)

    def __getitem__(self, idx: int) -> dict:
        item = {
            "prompt": self.prompts[idx],
            "options": [
                self.opt_a[idx], self.opt_b[idx], self.opt_c[idx], 
                self.opt_d[idx], self.opt_e[idx]
            ]
        }
        
        if not self.is_test:
            item["label"] = self.labels[idx]
            
        return item

In [ ]:
# Define MCQCollateFn for dynamic padding with MCQDataset
class MCQCollateFn:
    def __init__(self, tokenizer: BPETokenizer) -> None:
        self.pad_id = tokenizer.token_to_id("[pad]")

    def pad_sequence(self, sequences: list[dict], max_len: int) -> tuple[torch.Tensor, torch.Tensor]:
        padded_ids = []
        padded_masks = []
        
        for seq in sequences:
            pad_len = max_len - len(seq["input_ids"])
            padded_ids.append(seq["input_ids"] + [self.pad_id] * pad_len)
            padded_masks.append(seq["attention_mask"] + [0] * pad_len)
            
        return torch.tensor(padded_ids, dtype=torch.long), torch.tensor(padded_masks, dtype=torch.long)

    def __call__(self, batch: list[dict]) -> dict:
        # Pad the Prompts
        prompts = [item["prompt"] for item in batch]
        prompt_max_len = max(len(p["input_ids"]) for p in prompts)
        prompt_ids, prompt_masks = self.pad_sequence(prompts, prompt_max_len)
        
        # Pad the Options (Flatten the 5 options across the batch to find global max for options)
        all_options = [opt for item in batch for opt in item["options"]]
        opt_max_len = max(len(o["input_ids"]) for o in all_options)
        
        # Pad all options, then reshape them back into (Batch, 5, Max_Opt_Len)
        flat_opt_ids, flat_opt_masks = self.pad_sequence(all_options, opt_max_len)
        batch_size = len(batch)
        
        opt_ids = flat_opt_ids.view(batch_size, 5, opt_max_len)
        opt_masks = flat_opt_masks.view(batch_size, 5, opt_max_len)
        
        collated = {
            "prompt_ids": prompt_ids,          # (Batch, Seq_P)
            "prompt_mask": prompt_masks,       # (Batch, Seq_P)
            "opt_ids": opt_ids,                # (Batch, 5, Seq_O)
            "opt_mask": opt_masks              # (Batch, 5, Seq_O)
        }
        
        if "label" in batch[0]:
            collated["labels"] = torch.tensor([item["label"] for item in batch], dtype=torch.long)
            
        return collated

In [ ]:
# Define custom BiLSTM Cross Attention Classifier
class BiLSTMCrossAttentionClassifier(nn.Module):
    def __init__(
        self, 
        vocab_size: int = 2000, 
        embedding_dim: int = 64, 
        num_layers: int = 1,
        hidden_dim: int = 64, 
        dropout: float = 0.4
    ) -> None:
        
        super().__init__()

        # Map token IDs -> dense vectors; ignore padding index 0
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # Encoder Layer
        self.encoder = nn.LSTM(
            input_size=embedding_dim, 
            hidden_size=hidden_dim, 
            num_layers=num_layers, 
            batch_first=True, 
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Cross-Attention Projection Matrices
        # Option represents Query; Prompt represents Key and Value
        self.W_q = nn.Linear(hidden_dim * 2, hidden_dim * 2)   # Projects option -> Query space
        self.W_k = nn.Linear(hidden_dim * 2, hidden_dim * 2)   # Projects prompt -> Key space
        
        # Fusion and Classification Head
        # Input to fusion: [Option_Max_Pool, Context, Option_Max_Pool * Context]
        # Size: (hidden_dim*2) * 3 = hidden_dim*6
        fusion_dim = hidden_dim * 6
        
        # Classifies fused representation into scalar relevance score per option
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, hidden_dim),  # Compress fusion dim -> hidden dim
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),         # Stabilize activations across batch
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)            # Single score per option
        )

    def forward(
        self, 
        prompt_ids: torch.Tensor, 
        prompt_mask: torch.Tensor, 
        opt_ids: torch.Tensor, 
        opt_mask: torch.Tensor
    ) -> torch.Tensor:
        
        B = prompt_ids.size(0)
        
        # Encode prompt: token IDs -> Encoder hidden states
        p_emb = self.embedding(prompt_ids)
        p_out, _ = self.encoder(p_emb)
        
        # Flatten all 5 options into single batch dimension for parallel encoding
        Seq_O = opt_ids.size(2)
        flat_opt_ids = opt_ids.view(B * 5, Seq_O)
        
        # Encode all options in single Encoder pass
        o_emb = self.embedding(flat_opt_ids)
        o_out, _ = self.encoder(o_emb)
        
        # Restore batch and option dimensions after encoding
        o_out = o_out.view(B, 5, Seq_O, -1)
        
        # Broadcast prompt across all 5 options for simultaneous cross-attention
        p_out_expanded = p_out.unsqueeze(1).expand(B, 5, p_out.size(1), -1)
        p_mask_expanded = prompt_mask.unsqueeze(1).expand(B, 5, prompt_mask.size(1))
        
        # Project into Query, Key, Value spaces for cross-attention
        Q = self.W_q(o_out)
        K = self.W_k(p_out_expanded)
        V = p_out_expanded

        # Scaled dot-product attention: each option token attends over all prompt tokens
        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(K.size(-1))
        
        # Suppress prompt padding positions
        mask = p_mask_expanded.unsqueeze(2).expand_as(scores)
        scores = scores.masked_fill(mask == 0, -1e9)
        
        # Normalize scores -> attention weights over prompt positions
        attn_weights = F.softmax(scores, dim=-1)
        
        # Weighted sum of prompt values - context each option token extracts from prompt
        context = torch.matmul(attn_weights, V)
        
        # Mask padding positions in options before pooling - prevent padding from winning max
        opt_mask_exp = opt_mask.unsqueeze(-1).expand_as(o_out)
        o_out_masked = o_out.masked_fill(opt_mask_exp == 0, -1e9)
        context_masked = context.masked_fill(opt_mask_exp == 0, -1e9)
        
        # Max pooling - retain strongest signal across option sequence
        o_pooled = o_out_masked.max(dim=2)[0]
        c_pooled = context_masked.max(dim=2)[0]
        
        # Element-wise product captures feature-level alignment between option and prompt
        interaction = o_pooled * c_pooled
        
        # Concatenate all three views into rich 384-dim representation per option
        fused = torch.cat([o_pooled, c_pooled, interaction], dim=-1)
        
        # Flatten options into batch for linear layer compatibility
        fused_flat = fused.view(B * 5, -1)
        
        # Score each option independently
        logits_flat = self.fc(fused_flat)
        
        # Reshape -> one score per option per sample; argmax selects best option
        logits = logits_flat.view(B, 5)
        
        return logits

In [ ]:
# Define custom BiGRU Cross Attention Classifier
class BiGRUCrossAttentionClassifier(nn.Module):
    def __init__(
        self, 
        vocab_size: int = 2000, 
        embedding_dim: int = 64, 
        num_layers: int = 1,
        hidden_dim: int = 64, 
        dropout: float = 0.4
    ) -> None:
        
        super().__init__()

        # Map token IDs -> dense vectors; ignore padding index 0
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # Encoder Layer
        self.encoder = nn.GRU(
            input_size=embedding_dim, 
            hidden_size=hidden_dim, 
            num_layers=num_layers, 
            batch_first=True, 
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Cross-Attention Projection Matrices
        # Option represents Query; Prompt represents Key and Value
        self.W_q = nn.Linear(hidden_dim * 2, hidden_dim * 2)   # Projects option -> Query space
        self.W_k = nn.Linear(hidden_dim * 2, hidden_dim * 2)   # Projects prompt -> Key space
        
        # Fusion and Classification Head
        # Input to fusion: [Option_Max_Pool, Context, Option_Max_Pool * Context]
        # Size: (hidden_dim*2) * 3 = hidden_dim*6
        fusion_dim = hidden_dim * 6
        
        # Classifies fused representation into scalar relevance score per option
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, hidden_dim),  # Compress fusion dim -> hidden dim
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),         # Stabilize activations across batch
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)            # Single score per option
        )

    
    def forward(
        self, 
        prompt_ids: torch.Tensor, 
        prompt_mask: torch.Tensor, 
        opt_ids: torch.Tensor, 
        opt_mask: torch.Tensor
    ) -> torch.Tensor:
        
        B = prompt_ids.size(0)
        
        # Encode prompt: token IDs -> Encoder hidden states
        p_emb = self.embedding(prompt_ids)
        p_out, _ = self.encoder(p_emb)
        
        # Flatten all 5 options into single batch dimension for parallel encoding
        Seq_O = opt_ids.size(2)
        flat_opt_ids = opt_ids.view(B * 5, Seq_O)
        
        # Encode all options in single Encoder pass
        o_emb = self.embedding(flat_opt_ids)
        o_out, _ = self.encoder(o_emb)
        
        # Restore batch and option dimensions after encoding
        o_out = o_out.view(B, 5, Seq_O, -1)
        
        # Broadcast prompt across all 5 options for simultaneous cross-attention
        p_out_expanded = p_out.unsqueeze(1).expand(B, 5, p_out.size(1), -1)
        p_mask_expanded = prompt_mask.unsqueeze(1).expand(B, 5, prompt_mask.size(1))
        
        # Project into Query, Key, Value spaces for cross-attention
        Q = self.W_q(o_out)
        K = self.W_k(p_out_expanded)
        V = p_out_expanded

        # Scaled dot-product attention: each option token attends over all prompt tokens
        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(K.size(-1))
        
        # Suppress prompt padding positions
        mask = p_mask_expanded.unsqueeze(2).expand_as(scores)
        scores = scores.masked_fill(mask == 0, -1e9)
        
        # Normalize scores -> attention weights over prompt positions
        attn_weights = F.softmax(scores, dim=-1)
        
        # Weighted sum of prompt values - context each option token extracts from prompt
        context = torch.matmul(attn_weights, V)
        
        # Mask padding positions in options before pooling - prevent padding from winning max
        opt_mask_exp = opt_mask.unsqueeze(-1).expand_as(o_out)
        o_out_masked = o_out.masked_fill(opt_mask_exp == 0, -1e9)
        context_masked = context.masked_fill(opt_mask_exp == 0, -1e9)
        
        # Max pooling - retain strongest signal across option sequence
        o_pooled = o_out_masked.max(dim=2)[0]
        c_pooled = context_masked.max(dim=2)[0]
        
        # Element-wise product captures feature-level alignment between option and prompt
        interaction = o_pooled * c_pooled
        
        # Concatenate all three views into rich 384-dim representation per option
        fused = torch.cat([o_pooled, c_pooled, interaction], dim=-1)
        
        # Flatten options into batch for linear layer compatibility
        fused_flat = fused.view(B * 5, -1)
        
        # Score each option independently
        logits_flat = self.fc(fused_flat)
        
        # Reshape -> one score per option per sample; argmax selects best option
        logits = logits_flat.view(B, 5)
        
        return logits

In [ ]:
# Define function to compute Mean Average Precision@3 (MAP@3)
def compute_map3(scores: np.ndarray, df: pl.DataFrame) -> float:
    scores = np.asarray(scores).reshape(len(df), 5)
    true = df["label_id"].to_numpy()

    top3 = np.argsort(-scores, axis=1)[:, :3]
    hit = top3 == true[:, None]
    ranks = np.where(hit.any(axis=1), hit.argmax(axis=1) + 1, 0)

    out = np.zeros_like(ranks, dtype=float)
    np.divide(1.0, ranks, out=out, where=ranks > 0)
    
    return float(out.mean())

In [ ]:
# Define function to compute out-of-fold probabilities
def compute_oof_probs(model: nn.Module, loader: DataLoader, DEVICE: torch.device | str) -> np.ndarray:
    model.eval()
    start_idx = 0
    num_samples = len(loader.dataset) # type: ignore
    all_probs = torch.zeros((num_samples, 5), device=DEVICE)
    
    with torch.no_grad():
        for batch in loader:
            prompt_ids = batch["prompt_ids"].to(DEVICE)
            prompt_mask = batch["prompt_mask"].to(DEVICE)
            opt_ids = batch["opt_ids"].to(DEVICE)
            opt_mask = batch["opt_mask"].to(DEVICE)
            
            logits = model(prompt_ids, prompt_mask, opt_ids, opt_mask)

            probs = F.softmax(logits, dim=1)
            
            batch_size = probs.size(0)
            all_probs[start_idx : start_idx + batch_size] = probs
            start_idx += batch_size
            
    return all_probs.cpu().numpy()

In [ ]:
# Get embeddings for dimensionality reduction and clustering
embed_model = SentenceTransformer(EMBED_MODEL).to(DEVICE)
embeddings = embed_model.encode(
    train_data["mcq_query"].to_list(), 
    show_progress_bar=True,
    batch_size=16
)

del embed_model
gc.collect()

In [ ]:
# Reduce high-dimensional embeddings to 10 dimensions using PaCMAP for clustering
pacmap = PaCMAP(
    n_components=10,
    n_neighbors=30,
    apply_pca=True,
    distance="euclidean",
    random_state=SEED
)

pacmap_data = pacmap.fit_transform(embeddings)

# Find clusters from 10 dimensional embeddings for better cross-validation
clusterer = HDBSCAN(
    min_cluster_size=5,
    min_samples=3,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

cluster_labels = clusterer.fit_predict(pacmap_data) # type: ignore

valid_clusters = np.array(cluster_labels)

In [ ]:
# Check cluster information
n_clusters = len(set(cluster_labels))
print("Number of clusters:", n_clusters)

cluster_sizes = (
    pl.DataFrame({"cluster": cluster_labels})
    .group_by("cluster")
    .len()
    .sort("len", descending=True)
)
print(cluster_sizes)

# Add cluster information to train datafrane
train_data = train_data.with_columns(
    pl.Series("cluster", cluster_labels)
)

In [ ]:
# Define function for Hyper Parameter Tuning for BiLSTM model with `Qwen3-Embedding-0.6B` clusters
def objective(trial):
    # Define hyperparameter space to search through
    opt_vocab_size = trial.suggest_int("vocab_size", 2000, 3000)

    opt_embedding_dim = trial.suggest_categorical("embedding_dim", [32, 64, 128, 192, 256])
    opt_hidden_dim = trial.suggest_categorical("hidden_dim", [32, 64, 128, 192, 256])
    opt_dropout = trial.suggest_float("dropout", 0.25, 0.5)
        
    opt_learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)
    opt_weight_decay = trial.suggest_float("weight_decay", 1e-2, 5e-1, log=True)

    # Instantiate and train tokenizer and custom collate function
    tokenizer = BPETokenizer(vocab_size=opt_vocab_size, min_frequency=1)
    
    tokenizer_data = tokenizer.build_context(train_data)
    tokenizer.train(tokenizer_data)
    
    mcq_collate_fn = MCQCollateFn(tokenizer)
    
    # Configure CV constructor, and dummy features
    gkf = GroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED) # type: ignore
    
    X = np.zeros(len(train_data))
    y = np.zeros(len(train_data))
    groups = train_data["cluster"].fill_null(-1).to_numpy()
    
    oof_probs = np.zeros((len(train_data), 5), dtype=np.float32)
    best_fold_paths = []

    # Start iterating through {N_SPLITS}
    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):    
        train_fold = train_data[train_idx]
        val_fold = train_data[val_idx]
    
        # Configure DataLoaders for current fold
        train_loader = DataLoader(
            MCQDataset(
                train_fold, 
                tokenizer, 
                is_test=False
            ),
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=4,
            pin_memory=True,
            worker_init_fn=seed_worker,
            generator=train_generator,
            collate_fn=mcq_collate_fn
        )
    
        val_loader = DataLoader(
            MCQDataset(
                val_fold,
                tokenizer, 
                is_test=False
            ),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=4,
            pin_memory=True,
            worker_init_fn=seed_worker,
            generator=val_generator,
            collate_fn=mcq_collate_fn
        )
    
        # Configure model for current fold
        model = BiLSTMCrossAttentionClassifier(
            vocab_size=tokenizer.get_vocab_size(),
            embedding_dim=opt_embedding_dim,
            hidden_dim=opt_hidden_dim,
            dropout=opt_dropout
        )
        
        model = model.to(DEVICE)
    
        # Configure optimizer, and loss function for current fold
        optimizer = optim.AdamW(
            model.parameters(),
            lr=opt_learning_rate,
            weight_decay=opt_weight_decay
        )
        criterion = nn.CrossEntropyLoss()
    
        # Configure best validation score and best model path for current fold
        best_val_map3 = -1.0
        best_path = os.path.join(BEST_DIR, f"f_{fold+1}_best.pt")
        best_fold_paths.append(best_path)
    
        # Start training loop for current fold
        for epoch in range(EPOCHS):    
            model.train()
            train_loss_sum = 0.0
    
            optimizer.zero_grad()
            
            for step, batch in enumerate(train_loader):
                prompt_ids = batch["prompt_ids"].to(DEVICE)
                prompt_mask = batch["prompt_mask"].to(DEVICE)
                opt_ids = batch["opt_ids"].to(DEVICE)
                opt_mask = batch["opt_mask"].to(DEVICE)
                labels = batch["labels"].to(DEVICE)
    
                # Forward pass
                train_logits = model(prompt_ids, prompt_mask, opt_ids, opt_mask)
    
                # Compute standard loss
                loss = criterion(train_logits, labels)
                
                # Scale the loss
                steps_to_accumulate = min(ACCUMULATION_STEPS, len(train_loader) - (step // ACCUMULATION_STEPS) * ACCUMULATION_STEPS)
                scaled_loss = loss / steps_to_accumulate
                scaled_loss.backward()
                
                # Optimizer step and zero gradients when `ACCUMULATION_STEPS` is reached, or at end of epoch
                if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
                    optimizer.step()
                    optimizer.zero_grad()
                    
                train_loss_sum += loss.item() * prompt_ids.size(0)
                
            train_loss = train_loss_sum / len(train_fold)
    
            model.eval()
            val_loss_sum = 0.0
    
            val_start_idx = 0
            num_val_samples = len(val_loader.dataset) # type: ignore
            all_val_probs = torch.zeros((num_val_samples, 5), device=DEVICE)
    
            # Compute validation scores for current epoch
            with torch.no_grad():
                for batch in val_loader:
                    prompt_ids = batch["prompt_ids"].to(DEVICE)
                    prompt_mask = batch["prompt_mask"].to(DEVICE)
                    opt_ids = batch["opt_ids"].to(DEVICE)
                    opt_mask = batch["opt_mask"].to(DEVICE)
                    labels = batch["labels"].to(DEVICE)
                    
                    val_logits = model(prompt_ids, prompt_mask, opt_ids, opt_mask)
                    
                    loss = criterion(val_logits, labels)
                    val_loss_sum += loss.item() * prompt_ids.size(0)
    
                    val_probs = F.softmax(val_logits, dim=1)
    
                    batch_size = val_probs.size(0)
                    all_val_probs[val_start_idx : val_start_idx + batch_size] = val_probs
                    val_start_idx += batch_size
                    
            all_val_probs = all_val_probs.cpu().numpy()
            
            val_map3 = compute_map3(all_val_probs, val_fold)
        
            # Save best model per fold
            if val_map3 > best_val_map3:
                best_val_map3 = val_map3
                state = model.state_dict()
                torch.save(state, best_path)
    
        del model
        gc.collect()
    
        # Load saved best model for current fold to compute out-of-folds probabilities
        model = BiLSTMCrossAttentionClassifier(
            vocab_size=tokenizer.get_vocab_size(),
            embedding_dim=opt_embedding_dim,
            hidden_dim=opt_hidden_dim,
            dropout=opt_dropout
        )
    
        state = torch.load(best_path, map_location="cpu")
        model.load_state_dict(state)
        model = model.to(DEVICE)
    
        best_val_probs = compute_oof_probs(model, val_loader, DEVICE)
        oof_probs[val_idx] = best_val_probs
        
        del model, optimizer
        torch.cuda.empty_cache()
        gc.collect()
    
    oof_map3 = compute_map3(oof_probs, train_data)
    print(f"Out-of-fold MAP@3 score: {oof_map3:.8f}")

    return oof_map3

In [ ]:
# # Run Optuna Study for best BiLSTM Hyperparameter Search
# storage_name = "sqlite:///bilstm_tune_study.db"
# sampler = TPESampler(
#     n_startup_trials=5,
#     multivariate=True, 
#     seed=SEED
# )
    
# study = optuna.create_study(
#     study_name="bilstm_tune",
#     direction="maximize",
#     sampler=sampler,
#     storage=storage_name,
#     load_if_exists=True
# )
    
# study.optimize(objective, n_trials=50, show_progress_bar=True)

# trial = study.best_trial
    
# print("Best OOF MAP@3:", trial.value)
# print("Best Params:")
# for key, value in trial.params.items():
#     print(f"\t> {key}: {value}")

In [ ]:
# Best OOF MAP@3: 0.64625
# Best Params:
# 	> vocab_size: 2191
# 	> embedding_dim: 192
# 	> hidden_dim: 128
# 	> dropout: 0.37321880243176103
# 	> learning_rate: 0.00048606151236654744
# 	> weight_decay: 0.012109982578651377

opt_vocab_size = 2191

opt_embedding_dim = 192
opt_hidden_dim = 128
opt_dropout = 0.37321880243176103

opt_learning_rate = 0.00048606151236654744
opt_weight_decay = 0.012109982578651377

In [ ]:
# Instantiate and train tokenizer and custom collate function
# tokenizer = BPETokenizer(vocab_size=VOCAB_SIZE, min_frequency=1)
tokenizer = BPETokenizer(vocab_size=opt_vocab_size, min_frequency=1)

tokenizer_data = tokenizer.build_context(train_data)
tokenizer.train(tokenizer_data)

mcq_collate_fn = MCQCollateFn(tokenizer)

# Configure CV constructor, and dummy features
gkf = GroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED) # type: ignore

X = np.zeros(len(train_data))
y = np.zeros(len(train_data))
groups = train_data["cluster"].fill_null(-1).to_numpy()

oof_probs = np.zeros((len(train_data), 5), dtype=np.float32)
best_fold_paths = []

# Configure WandB logging
# run = wandb.init(
#     entity="24f2005537-dl-genai-project",
#     project="dl-genai-project",
#     name=WANDB_RUN_NAME,
#     config={
#         "model": {
#             "name": RANK_MODEL,
#             "num_layers": NUM_ENCODER_LAYERS,
#             "embedding_dim": opt_embedding_dim,
#             "hidden_dim": opt_hidden_dim,
#             "dropout": opt_dropout
#         },
#         "training": {
#             "splits": N_SPLITS,
#             "epochs": EPOCHS,
#             "optimizer": "AdamW",
#             "lr": opt_learning_rate,
#             "weight_decay": opt_weight_decay,
#             "loss": "CrossEntropyLoss",
#             "scheduler": None
#         },
#         "data": {
#             "batch_size": BATCH_SIZE,
#             "grad_accumulation_steps": ACCUMULATION_STEPS,
#             "vocab_size": opt_vocab_size, 
#             "embed_model": EMBED_MODEL
#         }
#     }
# )

# Start iterating through {N_SPLITS}
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):

    fold_start = time.perf_counter()
    print(f"FOLD {fold+1}/{N_SPLITS} | "
          f"Train Set Size: {len(train_idx)} | Val Set Size: {len(val_idx)}\n")

    train_fold = train_data[train_idx]
    val_fold = train_data[val_idx]

    # Configure DataLoaders for current fold
    train_loader = DataLoader(
        MCQDataset(
            train_fold, 
            tokenizer, 
            is_test=False
        ),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=train_generator,
        collate_fn=mcq_collate_fn
    )

    val_loader = DataLoader(
        MCQDataset(
            val_fold,
            tokenizer, 
            is_test=False
        ),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=val_generator,
        collate_fn=mcq_collate_fn
    )

    # Configure model for current fold
    model = BiLSTMCrossAttentionClassifier(
        vocab_size=tokenizer.get_vocab_size(),
        embedding_dim=opt_embedding_dim,
        hidden_dim=opt_hidden_dim,
        dropout=opt_dropout
    )
    # model = BiGRUCrossAttentionClassifier(
    #     vocab_size=tokenizer.get_vocab_size(),
    #     embedding_dim=EMBEDDING_DIM,
    #     num_layers=NUM_ENCODER_LAYERS,
    #     hidden_dim=HIDDEN_DIM,
    #     dropout=DROPOUT
    # )
    
    model = model.to(DEVICE)

    # Configure optimizer, and loss function for current fold
    # optimizer = optim.AdamW(
    #     model.parameters(),
    #     lr=5e-4,
    #     weight_decay=0.01
    # )
    optimizer = optim.AdamW(
        model.parameters(),
        lr=opt_learning_rate,
        weight_decay=opt_weight_decay
    )
    criterion = nn.CrossEntropyLoss()

    # Configure best validation score and best model path for current fold
    best_val_map3 = -1.0
    best_path = os.path.join(BEST_DIR, f"f_{fold+1}_best.pt")
    best_fold_paths.append(best_path)

    # Start training loop for current fold
    for epoch in range(EPOCHS):
        epoch_start = time.perf_counter()

        model.train()
        train_loss_sum = 0.0

        optimizer.zero_grad()
        
        for step, batch in enumerate(train_loader):
            prompt_ids = batch["prompt_ids"].to(DEVICE)
            prompt_mask = batch["prompt_mask"].to(DEVICE)
            opt_ids = batch["opt_ids"].to(DEVICE)
            opt_mask = batch["opt_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            # Forward pass
            train_logits = model(prompt_ids, prompt_mask, opt_ids, opt_mask)

            # Compute standard loss
            loss = criterion(train_logits, labels)
            
            # Scale the loss
            steps_to_accumulate = min(ACCUMULATION_STEPS, len(train_loader) - (step // ACCUMULATION_STEPS) * ACCUMULATION_STEPS)
            scaled_loss = loss / steps_to_accumulate
            scaled_loss.backward()
            
            # Optimizer step and zero gradients when `ACCUMULATION_STEPS` is reached, or at end of epoch
            if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
                optimizer.step()
                optimizer.zero_grad()
                
            train_loss_sum += loss.item() * prompt_ids.size(0)
            
        train_loss = train_loss_sum / len(train_fold)

        model.eval()
        val_loss_sum = 0.0

        val_start_idx = 0
        num_val_samples = len(val_loader.dataset) # type: ignore
        all_val_probs = torch.zeros((num_val_samples, 5), device=DEVICE)

        # Compute validation scores for current epoch
        with torch.no_grad():
            for batch in val_loader:
                prompt_ids = batch["prompt_ids"].to(DEVICE)
                prompt_mask = batch["prompt_mask"].to(DEVICE)
                opt_ids = batch["opt_ids"].to(DEVICE)
                opt_mask = batch["opt_mask"].to(DEVICE)
                labels = batch["labels"].to(DEVICE)
                
                val_logits = model(prompt_ids, prompt_mask, opt_ids, opt_mask)
                
                loss = criterion(val_logits, labels)
                val_loss_sum += loss.item() * prompt_ids.size(0)

                val_probs = F.softmax(val_logits, dim=1)

                batch_size = val_probs.size(0)
                all_val_probs[val_start_idx : val_start_idx + batch_size] = val_probs
                val_start_idx += batch_size
                
        all_val_probs = all_val_probs.cpu().numpy()
        
        val_loss = val_loss_sum / num_val_samples
        val_map3 = compute_map3(all_val_probs, val_fold)

        epoch_time = time.perf_counter() - epoch_start

        # run.log({
        #     "epoch_secs": round(epoch_time, 4),
        #     "train_loss": train_loss,
        #     "val_loss": val_loss, "val_map3": val_map3
        # })
        print(
            f"Fold: {fold+1}/{N_SPLITS} Epoch: {epoch+1}/{EPOCHS} Time: {epoch_time:.2f}s\n"
            f"\tTrain Loss: {train_loss:.6f}\n"
            f"\tVal Loss: {val_loss:.6f} | Val MAP@3: {val_map3:.6f}"
        )

        # Save best model per fold
        if val_map3 > best_val_map3:
            best_val_map3 = val_map3
            state = model.state_dict()
            
            torch.save(state, best_path)

    del model
    gc.collect()

    # Load saved best model for current fold to compute out-of-folds probabilities
    model = BiLSTMCrossAttentionClassifier(
        vocab_size=tokenizer.get_vocab_size(),
        embedding_dim=opt_embedding_dim,
        hidden_dim=opt_hidden_dim,
        dropout=opt_dropout
    )
    # model = BiGRUCrossAttentionClassifier(
    #     vocab_size=tokenizer.get_vocab_size(),
    #     embedding_dim=EMBEDDING_DIM,
    #     num_layers=NUM_ENCODER_LAYERS,
    #     hidden_dim=HIDDEN_DIM,
    #     dropout=DROPOUT
    # )

    state = torch.load(best_path, map_location="cpu")
    model.load_state_dict(state)
    model = model.to(DEVICE)

    best_val_probs = compute_oof_probs(model, val_loader, DEVICE)
    oof_probs[val_idx] = best_val_probs

    fold_time = time.perf_counter() - fold_start

    # run.log({
    #     "fold_secs": round(fold_time, 4),
    #     "best_val_map3": best_val_map3
    # })
    print(
        f"\nFold {fold+1} completed in {int(fold_time // 60)} min {fold_time % 60:.2f}s"
        f" | Best Val MAP@3: {best_val_map3:.6f}\n"
    )

    del model, optimizer
    torch.cuda.empty_cache()
    gc.collect()

# run.finish()
oof_map3 = compute_map3(oof_probs, train_data)
print(f"\nOut-of-fold MAP@3 score: {oof_map3:.8f}")

OOF Scores: 

- 0.60300000 (0.73815) Bi-LSTM [all-MiniLM-L6-V2]
- 0.60483333 (0.74688) Bi-LSTM [bge-m3]
- 0.60591667 (0.74937) Bi-LSTM [bge-m3] Hypertuned
---
- 0.58291667 (0.XXXXX) Bi-GRU [all-MiniLM-L6-V2]
- 0.58866667 (0.74064) Bi-GRU [bge-m3]

In [ ]:
# # Inference code for given custom model trained with CrossEntropyLoss
# test_scores = np.zeros((len(test_data), 5))

# test_loader = DataLoader(
#         MCQDataset(
#             test_data,
#             tokenizer,
#             is_test=True
#         ),
#         batch_size=BATCH_SIZE,
#         shuffle=False,
#         num_workers=4,
#         pin_memory=True,
#         worker_init_fn=seed_worker,
#         generator=test_generator,
#         collate_fn=mcq_collate_fn
#     )

# for fold, best_path in enumerate(best_fold_paths):
#     model = BiLSTMCrossAttentionClassifier(
#         vocab_size=tokenizer.get_vocab_size(),
#         embedding_dim=opt_embedding_dim,
#         num_layers=opt_num_layers,
#         hidden_dim=opt_hidden_dim,
#         dropout=opt_dropout
#     )
#     # model = BiGRUCrossAttentionClassifier(
#     #     vocab_size=tokenizer.get_vocab_size(),
#     #     embedding_dim=EMBEDDING_DIM,
#     #     num_layers=NUM_ENCODER_LAYERS,
#     #     hidden_dim=HIDDEN_DIM,
#     #     dropout=DROPOUT
#     # )

#     state = torch.load(best_path, map_location="cpu")
#     model.load_state_dict(state)
#     model = model.to(DEVICE)
#     model.eval()
    
#     test_start_idx = 0
#     num_test_samples = len(test_loader.dataset) # type: ignore
#     all_test_probs = torch.zeros((num_test_samples, 5), device=DEVICE)

#     with torch.no_grad():
#         for batch in test_loader:
#             prompt_ids = batch["prompt_ids"].to(DEVICE)
#             prompt_mask = batch["prompt_mask"].to(DEVICE)
#             opt_ids = batch["opt_ids"].to(DEVICE)
#             opt_mask = batch["opt_mask"].to(DEVICE)
                
#             test_logits = model(prompt_ids, prompt_mask, opt_ids, opt_mask)

#             test_probs = F.softmax(test_logits, dim=1)

#             batch_size = test_probs.size(0)
#             all_test_probs[test_start_idx : test_start_idx + batch_size] = test_probs
#             test_start_idx += batch_size

#     all_test_probs = all_test_probs.cpu().numpy()

#     test_scores += all_test_probs / len(best_fold_paths)

#     del model
#     torch.cuda.empty_cache()
#     gc.collect()

# top3_idx = np.argsort(-test_scores, axis=1)[:, :3]
# pred_strings = [" ".join(OPTION_COLS[i] for i in row) for row in top3_idx]

# submission = pl.DataFrame({"ID": test_data[ID_COL], "Prediction": pred_strings})
# submission.write_csv("submission.csv")
# print(submission.sample(5))